# EU Electricity Prices Data Cleaning Script

This notebook reads a CSV file of EU electricity prices, performs data cleaning, and writes a cleaned output file.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Step 1: Create Sample EU Electricity Prices Data

First, let's create a sample CSV file with EU electricity prices that might have quality issues.

In [ ]:
# Create sample data with common data quality issues
sample_data = '''Date,Country,Price_EUR_per_MWh,Volume_MWh,Supplier,Status
2024-01-01,Germany,85.50,1500,TenneT,confirmed
2024-01-01,France,92.30,2000,RTE,confirmed
2024-01-01,Germany,85.50,1500,TenneT,confirmed
2024-01-02,Spain,,1200,REE,confirmed
2024-01-02,Italy,88.75,999999,TERNA,confirmed
2024-01-02,Germany,invalid,1400,TenneT,pending
2024-01-03,France,91.20,1800,RTE,confirmed
2024-01-03,Germany,86.90,1600,,confirmed
2024-01-04,Spain,90.10,1300,REE,cancelled
2024-01-04,Italy,87.40,1100,TERNA,confirmed
2024-01-05,France,89.50,1900,RTE,confirmed
2024-01-05,Germany,84.20,1550,TenneT,confirmed'''

# Write to CSV file using 'with' statement
with open('raw_electricity_prices.csv', 'w', encoding='utf-8') as f:
    f.write(sample_data)

print("✓ Sample data file created: raw_electricity_prices.csv")

## Step 2: Read and Inspect Raw Data

In [ ]:
# Read CSV using 'with' statement for explicit file handling
with open('raw_electricity_prices.csv', 'r', encoding='utf-8') as f:
    df = pd.read_csv(f)

print("Raw Data Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

## Step 3: Data Cleaning Process

Apply the following cleaning steps:
1. Remove duplicate rows
2. Remove rows with missing critical values
3. Fix data type conversions
4. Remove outliers (unrealistic volume values)
5. Filter only confirmed records
6. Remove rows with invalid price data
7. Sort by date and country

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

print(f"Starting rows: {len(df_clean)}")

# Step 1: Remove complete duplicates
df_clean = df_clean.drop_duplicates()
print(f"After removing duplicates: {len(df_clean)}")

# Step 2: Remove rows with missing critical columns
critical_cols = ['Date', 'Country', 'Price_EUR_per_MWh']
df_clean = df_clean.dropna(subset=critical_cols)
print(f"After removing missing critical values: {len(df_clean)}")

# Step 3: Convert Date column to datetime
df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce')

# Step 4: Convert Price to numeric, handle non-numeric values
df_clean['Price_EUR_per_MWh'] = pd.to_numeric(df_clean['Price_EUR_per_MWh'], errors='coerce')
df_clean = df_clean.dropna(subset=['Price_EUR_per_MWh'])
print(f"After converting prices and removing invalid entries: {len(df_clean)}")

# Step 5: Remove unrealistic volume outliers (e.g., volume > 10000 MWh)
df_clean = df_clean[df_clean['Volume_MWh'] <= 10000]
print(f"After removing volume outliers: {len(df_clean)}")

# Step 6: Filter only confirmed records
df_clean = df_clean[df_clean['Status'] == 'confirmed']
print(f"After filtering confirmed records only: {len(df_clean)}")

# Step 7: Fill missing supplier with 'Unknown'
df_clean['Supplier'] = df_clean['Supplier'].fillna('Unknown')

# Step 8: Sort by Date and Country
df_clean = df_clean.sort_values(['Date', 'Country']).reset_index(drop=True)

print(f"\nFinal cleaned dataset: {len(df_clean)} rows")

## Step 4: Review Cleaned Data

In [ ]:
print("Cleaned Data:")
print(df_clean)
print("\nData Types:")
print(df_clean.dtypes)
print("\nData Summary Statistics:")
print(df_clean[['Price_EUR_per_MWh', 'Volume_MWh']].describe())

## Step 5: Write Cleaned Data to CSV

In [ ]:
# Write cleaned data using 'with' statement
output_filename = 'cleaned_electricity_prices.csv'

with open(output_filename, 'w', encoding='utf-8') as f:
    df_clean.to_csv(f, index=False, date_format='%Y-%m-%d')

print(f"✓ Cleaned data written to: {output_filename}")
print(f"✓ Total rows processed: {len(df)}")
print(f"✓ Rows retained: {len(df_clean)}")
print(f"✓ Rows removed: {len(df) - len(df_clean)}")

## Step 6: Summary Report

In [ ]:
print("="*50)
print("DATA CLEANING SUMMARY REPORT")
print("="*50)
print(f"\nInput file: raw_electricity_prices.csv")
print(f"Output file: {output_filename}")
print(f"\nRecords by Country:")
print(df_clean['Country'].value_counts())
print(f"\nPrice Statistics (EUR/MWh):")
print(f"  Minimum: €{df_clean['Price_EUR_per_MWh'].min():.2f}")
print(f"  Maximum: €{df_clean['Price_EUR_per_MWh'].max():.2f}")
print(f"  Average: €{df_clean['Price_EUR_per_MWh'].mean():.2f}")
print(f"  Median:  €{df_clean['Price_EUR_per_MWh'].median():.2f}")
print(f"\nData Quality Checks:")
print(f"  Missing values: {df_clean.isnull().sum().sum()}")
print(f"  Duplicate rows: {df_clean.duplicated().sum()}")
print(f"  Date range: {df_clean['Date'].min()} to {df_clean['Date'].max()}")